# Chunking Experiments

Notebook for exploring different chunking strategies (`recursive`, `semantic`, `sentence`)
and their effect on retrieval quality.  Compare chunk size, overlap, and boundary heuristics
against a set of golden queries.


In [ ]:
# TODO: %load_ext autoreload
# TODO: %autoreload 2

# TODO: from src.ingestion.loader import PolicyDocumentLoader
# TODO: from src.ingestion.chunker import PolicyChunker
# TODO: import pandas as pd
# TODO: import matplotlib.pyplot as plt

In [ ]:
# TODO: Load raw documents
# loader = PolicyDocumentLoader(source_dir='data/raw')
# documents = loader.load()
# print(f'Loaded {len(documents)} documents')

In [ ]:
# TODO: Experiment 1 — vary chunk_size
# results = []
# for size in [256, 512, 1024]:
#     chunker = PolicyChunker(chunk_size=size, chunk_overlap=64)
#     chunks = chunker.split(documents)
#     results.append({'chunk_size': size, 'n_chunks': len(chunks)})
# pd.DataFrame(results)

In [ ]:
# TODO: Visualise chunk length distribution
# lengths = [len(c.page_content) for c in chunks]
# plt.hist(lengths, bins=30)
# plt.xlabel('Chunk length (chars)')
# plt.ylabel('Count')
# plt.title('Chunk Length Distribution')
# plt.show()

In [7]:
import sys
sys.path.append("..")  # so we can import from src/

from src.ingestion.loader import PolicyDocumentLoader
from src.ingestion.chunker import PolicyChunker

In [8]:
loader = PolicyDocumentLoader("../data/raw")
docs = loader.load()

print(f"Pages loaded: {len(docs)}")
print(f"First page metadata: {docs[0].metadata}")
print(f"First page preview:\n{docs[0].page_content[:300]}")

Skipping unsupported file type: ../data/raw/.gitkeep
Pages loaded: 96
First page metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2021-09-01T11:56:41-04:00', 'author': 'U.S. Office of Personnel Management', 'keywords': 'pay; leave; eregency preparedness; weather;', 'moddate': '2021-09-01T13:20:22-04:00', 'subject': 'Handbook on Pay and Leave Benefits for Federal Employees Affected by Severe Weather Conditions or Other Emergency Situations', 'title': 'Handbook on Pay and Leave Benefits for Federal Employees Affected by Severe Weather Conditions or Other Emergency Situations', 'source': '../data/raw/emergencybenefits.pdf', 'total_pages': 20, 'page': 0, 'page_label': 'i'}
First page preview:
UNITED STATES OFFICE OF PERSONNEL MANAGEMENT 
Handbook on Pay and Leave Benefts 
for Federal Employees 
Afected by Severe Weather Conditions 
or Other Emergency Situations 
OPM.GOV SEPTEMBER 2021


In [9]:
chunker = PolicyChunker(chunk_size=512, chunk_overlap=64, strategy="recursive")
chunks = chunker.split(docs)

print(f"Total chunks: {len(chunks)}")
print(f"\nChunk 0 metadata: {chunks[0].metadata}")
print(f"Chunk 0 content:\n{chunks[0].page_content}")

Total chunks: 478

Chunk 0 metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2021-09-01T11:56:41-04:00', 'author': 'U.S. Office of Personnel Management', 'keywords': 'pay; leave; eregency preparedness; weather;', 'moddate': '2021-09-01T13:20:22-04:00', 'subject': 'Handbook on Pay and Leave Benefits for Federal Employees Affected by Severe Weather Conditions or Other Emergency Situations', 'title': 'Handbook on Pay and Leave Benefits for Federal Employees Affected by Severe Weather Conditions or Other Emergency Situations', 'source': '../data/raw/emergencybenefits.pdf', 'total_pages': 20, 'page': 0, 'page_label': 'i', 'chunk_index': 0}
Chunk 0 content:
UNITED STATES OFFICE OF PERSONNEL MANAGEMENT 
Handbook on Pay and Leave Benefts 
for Federal Employees 
Afected by Severe Weather Conditions 
or Other Emergency Situations 
OPM.GOV SEPTEMBER 2021


I noticed cover pages and TOC sections were being indexed so I added a minimum content length filter to drop chunks below X characters.

In [10]:
chunker_char = PolicyChunker(chunk_size=512, chunk_overlap=64, strategy="character")
chunks_char = chunker_char.split(docs)

print(f"Recursive chunks: {len(chunks)}")
print(f"Character chunks: {len(chunks_char)}")


Recursive chunks: 478
Character chunks: 96


Compared recursive character splitting vs fixed character splitting on 96 pages of OPM compliance documents. Character splitting produced 96 chunks — one per page — exceeding the 512 token target. Recursive splitting produced 478 chunks by respecting paragraph and sentence boundaries, staying within the token budget while preserving semantic coherence

In [11]:
# inspect a mid-document recursive chunk to check quality
for chunk in chunks[10:15]:
    print("---")
    print(f"source: {chunk.metadata['source']}")
    print(f"page: {chunk.metadata['page']}")
    print(f"chunk_index: {chunk.metadata['chunk_index']}")
    print(f"length: {len(chunk.page_content)} chars")
    print(f"content:\n{chunk.page_content}")
    print()

---
source: ../data/raw/emergencybenefits.pdf
page: 3
chunk_index: 10
length: 462 chars
content:
2 
 
in advance of the date on which the employee otherwise would be entitled to be paid is 
required to help the employee defray immediate expenses incidental to the evacuation.  An 
authorized agency official must determine the time period (measured in days) to be used in 
computing the amount of the advance payment.  The selected time period may not exceed 
30 days.   
 
The agency must determine the days and hours the employee would have been expected to

---
source: ../data/raw/emergencybenefits.pdf
page: 3
chunk_index: 11
length: 442 chars
content:
work during the selected time period (but for the evacuation) as follows:  (1) for employees 
with a regularly scheduled tour of duty, the agency must determine the days and hours in 
the employee’s normal basic workweek during the selected time period; and (2) for 
intermittent employees, the agency must estimate the days and hours the emp

chunk 10 — it ends with "The agency must determine the days and hours the employee would have been expected to" and chunk 11 picks up with "work during the selected time period". That's a mid-sentence cut across a chunk boundary.
This is exactly what chunk_overlap=64 is supposed to help with but the overlap isn't showing here between chunk 10 and 11. The sentence is still broken.
This tells us 512 characters might be slightly small for dense government policy text where sentences run long. 

In [12]:
chunker_larger = PolicyChunker(chunk_size=1024, chunk_overlap=128, strategy="recursive")
chunks_larger = chunker_larger.split(docs)

print(f"512/64 chunks: {len(chunks)}")
print(f"1024/128 chunks: {len(chunks_larger)}")

# check same area
for chunk in chunks_larger[5:8]:
    print("---")
    print(f"length: {len(chunk.page_content)} chars")
    print(f"content:\n{chunk.page_content}")
    print()

512/64 chunks: 478
1024/128 chunks: 267
---
length: 333 chars
content:
must follow the regulations in 5 CFR part 550, subpart D, for evacuations from or within 
the United States and certain nonforeign areas.    
Advance Payments 
An agency may make an advance payment to an employee who has received an order to 
evacuate, provided that, in the opinion of the agency head or designated official, payment

---
length: 997 chars
content:
2 
 
in advance of the date on which the employee otherwise would be entitled to be paid is 
required to help the employee defray immediate expenses incidental to the evacuation.  An 
authorized agency official must determine the time period (measured in days) to be used in 
computing the amount of the advance payment.  The selected time period may not exceed 
30 days.   
 
The agency must determine the days and hours the employee would have been expected to 
work during the selected time period (but for the evacuation) as follows:  (1) for employees 
with 

I tested 512 vs 1024 chunk sizes on OPM policy documents and found that 512 characters cut mid-clause on dense government policy text. Since compliance answers require complete rule context to be accurate, I chose 1024 with 128 overlap — accepting higher token cost in exchange for answer reliability